[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/simulations/steady_state_and_uplift_increase.ipynb)

# Simulating a landscape: steady state, then an increase in uplift

*This lesson made by Simon M Mudd.*

In this notebook we use `lsdtt-lem`, the `lsdtt3` landscape evolution model, to build a synthetic mountain range. Rivers erode by the **stream power law** and hillslopes by **linear diffusion**, while the whole landscape is uplifted at a constant rate.

You will:

1. Make a starting surface: a gently tilted plane roughened with **red noise**.
2. "Snap" it to **steady state**, where erosion everywhere balances uplift, with a chosen relief.
3. Look at the steady-state landscape: hillshade, channel network, slope map, and the long profile and chi profile of the largest basin.
4. Increase the uplift rate and watch the landscape respond, with snapshots every 200,000 years.

### The model

Channels erode at a rate given by the stream power law,

$$E = K A^m S^n ,$$

where $A$ is drainage area, $S$ is channel slope and $K$ is the **erodibility**. We use $m = 0.45$ and $n = 1$. Pixels draining fewer than 20 pixels are hillslopes, which erode by linear diffusion,

$$\frac{\partial z}{\partial t} = U + D \nabla^2 z ,$$

where $D$ is the **hillslope diffusivity** and $U$ is the uplift rate.

At steady state, $E = U$ everywhere, so the stream power law gives the steady-state channel slope:

$$S = \left(\frac{U}{K}\right)^{1/n} A^{-m/n} = k_{sn} A^{-m/n} .$$

$k_{sn} = (U/K)^{1/n}$ is the **channel steepness index**. Faster uplift or less erodible rock both make steeper channels.

The `lsdtt3` programs are command-line tools controlled by small text **parameter files** (`key: value` lines). We write these files from Python and then run the programs with `!`.

## Setup

### First set up condacolab

This step takes around 2 minutes. **`condacolab.install()` restarts the Colab runtime.** You will see a message that the session crashed: this is expected. Wait for it to reconnect, then carry on with the next cell.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

Now install `pygmt` (GMT is the Generic Mapping Tools). This is the slowest step and takes around a minute.

In [ ]:
# Install pygmt. mamba prints hundreds of lines, so we hide them and only
# show the end of the output if something goes wrong.
import subprocess

result = subprocess.run(["mamba", "install", "-y", "pygmt"], capture_output=True, text=True)
if result.returncode == 0:
    print("pygmt installed successfully.")
else:
    print("pygmt installation FAILED. The last lines of the output were:\n")
    print("\n".join((result.stdout + result.stderr).splitlines()[-30:]))

Now get `lsdviztools3`, the plotting package.

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdviztools3-0.2.0-py3-none-any.whl

In [ ]:
!pip install "lsdviztools3-0.2.0-py3-none-any.whl[charts]"

Next, download and unpack the `lsdtt3` command-line programs.

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdtt3-backend-linux-x86_64-core-v0.7.1.tar.gz

In [ ]:
!tar -xzf lsdtt3-backend-linux-x86_64-core-v0.7.1.tar.gz

Tell the system where to find the `lsdtt3` programs and their libraries. If your runtime restarts later, run this cell again.

In [ ]:
import os

root = "/content/lsdtt3-backend-linux-x86_64-core-v0.7.1"
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's older proj.db breaks rasterio and pyproj.

Check that it works: this prints the version of `lsdtt-chi-analysis`.

In [ ]:
!lsdtt-chi-analysis -v

## Step 1: The model domain and the starting surface

The model grid is **150 pixels wide** (east–west) and **200 pixels long** (north–south), with 100 m pixels, so it covers 15 km × 20 km.

The edges are set with `boundary_conditions`, four letters for the north, east, south and west edges:

* `o` (open): a **fixed elevation** boundary, like a river flowing out to the sea or into a big trunk river that keeps its level.
* `n` (no flux): a closed edge. Nothing flows out of it.
* `p` (periodic): the edge wraps around. Water leaving the east edge comes back in on the west edge, as if the strip of landscape repeated side by side.

We use `npop`: the southern (150-pixel) edge is open at 0 m, the northern edge is closed, so it becomes the main drainage divide, and the east and west edges are periodic. All the water and sediment leaves through the south edge, at the bottom of the maps.

The starting surface is roughened with **red noise**. Red noise is random, but smooth: neighbouring pixels have similar elevations, so it has bumps and hollows several pixels across. Those bumps steer the first flow paths and become the seeds of the drainage network. On top of the noise we add two gentle slopes:

* `initial_slope`: a tilt down towards the open south edge.
* `valley_slope`: a tilt down towards the middle column, which makes a broad, shallow valley. The biggest river forms in this valley, so the largest basin sits in the middle of the model.

We make the noise with `lsdtt-lem` itself. A run of a single time step with erosion and uplift turned off just writes the surface out as `sim_init_initial_topography.tif`. Then we add the slopes in Python.

In [ ]:
prefix = "sim"

# Model grid
n_rows = 200          # north-south: the model is 200 pixels long
n_cols = 150          # east-west: the open (fixed elevation) edge is 150 pixels wide
dx = 100              # pixel size (m)
boundary_conditions = "npop"   # north, east, south, west: south open, north closed, east-west periodic

# Starting surface
initial_slope = 0.002          # tilt towards the open south edge (m/m)
valley_slope = 0.002           # tilt towards the middle column, making a valley (m/m)
noise_stddev_metres = 1.0      # size of the red-noise bumps (m)
random_seed = 1                # change this for a different random landscape

# Stream power and channels
m_exponent = 0.45
n_exponent = 1.0
threshold_pixels = 20          # pixels draining at least this many pixels are channels

with open(f"{prefix}_init.param", "w") as f:
    f.write(f"write_fname: {prefix}_init\n")
    f.write(f"n_rows: {n_rows}\n")
    f.write(f"n_cols: {n_cols}\n")
    f.write(f"cell_size_metres: {dx}\n")
    f.write(f"boundary_conditions: {boundary_conditions}\n")
    f.write("noise_type: red_noise\n")
    f.write(f"noise_target_stddev_metres: {noise_stddev_metres}\n")
    f.write("initial_surface_slope: 0\n")      # we add the slopes ourselves below
    f.write(f"random_seed: {random_seed}\n")
    # One time step with nothing switched on: this just writes the starting surface
    f.write("number_of_iterations: 1\n")
    f.write("snap_initial_surface_to_steady_state: false\n")
    f.write("enable_d8_fluvial_erosion: false\n")
    f.write("enable_uplift: false\n")

!lsdtt-lem ./ {prefix}_init.param > {prefix}_init.log
print("Wrote", f"{prefix}_init_initial_topography.tif")

### Add the slopes and fill the pits

Now we add the two slopes and set the open south edge to 0 m.

The noise leaves small pits: hollows with no way out. Water cannot flow out of a pit, so before we route flow we **fill** them, raising each pit until it just spills towards the open edge.

We do this in Python with a *priority-flood* fill (Barnes et al., 2014). It starts at the open edge and works inwards from the lowest pixel, wrapping around the periodic east and west edges, so that only the south edge is an outlet. (The pit filling built into this version of `lsdtt-lem` lets water escape through closed edges, which leaves pits on those edges that the model cannot drain.)

*Barnes, R., Lehman, C., Mulla, D., 2014. Priority-flood: An optimal depression-filling and watershed-labeling algorithm for digital elevation models. Computers & Geosciences 62, 117–127. https://doi.org/10.1016/j.cageo.2013.04.024*

In [ ]:
import heapq
import numpy as np
import rasterio


def fill_from_south(z, min_step=0.01):
    # Priority-flood fill with only the south (last row) edge as an outlet. The east
    # and west edges are periodic, so columns wrap around. Every pixel ends up at
    # least min_step metres above the pixel it drains to.
    nr, nc = z.shape
    z = z.copy()
    done = np.zeros(z.shape, bool)
    queue = []
    for c in range(nc):
        heapq.heappush(queue, (z[nr - 1, c], nr - 1, c))
        done[nr - 1, c] = True
    while queue:
        elev, r, c = heapq.heappop(queue)
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                rr, cc = r + dr, (c + dc) % nc
                if 0 <= rr < nr and not done[rr, cc]:
                    done[rr, cc] = True
                    z[rr, cc] = max(z[rr, cc], elev + min_step)
                    heapq.heappush(queue, (z[rr, cc], rr, cc))
    return z


with rasterio.open(f"{prefix}_init_initial_topography.tif") as src:
    noise = src.read(1).astype(float)
    profile = src.profile

rows_from_south = (n_rows - 1 - np.arange(n_rows))[:, None]   # the last row is the south edge
cols = np.arange(n_cols)[None, :]
z0 = (noise - noise.min()
      + initial_slope * dx * rows_from_south             # rises away from the south edge
      + valley_slope * dx * np.abs(cols - n_cols // 2))  # rises away from the middle column
z0[-1, :] = 0                                            # the open edge is at 0 m

z_filled = fill_from_south(z0)
print(f"Raised {(z_filled > z0 + 1e-6).sum()} of {z0.size} pixels to fill the pits.")

profile.update(dtype="float32", driver="GTiff")
with rasterio.open(f"{prefix}_initial_filled.tif", "w", **profile) as dst:
    dst.write(z_filled.astype("float32"), 1)

### Figure 1: The starting surface

The south (open) edge is at the bottom. The slopes add only a few tens of metres of relief, so the noise and the shallow central valley are both easy to see.

In [ ]:
from dataclasses import replace

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from lsdviztools3.io.vector import load_vector
from lsdviztools3.render.style import MapStyle
from lsdviztools3.render.hillshade import render_hillshade
from lsdviztools3.render.channels import render_drape, render_channels
from lsdviztools3.render.base import save_figure

style = MapStyle(figure_size="8c", transparent=False)
hs_style = replace(style, cmap="gray")

fig = render_hillshade(f"{prefix}_initial_filled.tif", shade=True, style=replace(style, cmap="gist_earth"))
save_figure(fig, f"{prefix}_initial_surface.png", style=style)
display(Image(f"{prefix}_initial_surface.png", width=350))

## Step 2: Snap the landscape to steady state

We could run the model forward in time until erosion balances uplift, but that takes millions of model years. Instead, `lsdtt-lem` can jump straight to steady state:

1. **Channels.** For a steady-state channel with $n = 1$, elevation rises linearly with the chi coordinate, $z = k_{sn} \chi$ (see Step 3). So once the flow paths are known, the steady-state channel elevations can be calculated directly. This is repeated (`steady_state_n_cycles`), re-routing the flow each time, so that the drainage divides can move.
2. **Hillslopes.** With the channels fixed, the hillslope pixels are solved for the shape at which diffusion exactly balances uplift (`run_hillslope_steady_state`, `hillslope_ss_model: linear`).

We ask for the **relief** we want rather than for $K$ (`steady_state_mode: relief`), and `lsdtt-lem` back-calculates the $K$ that gives it. `target_relief` sets the relief of the *channels*. The hillslopes then add some relief on top, so we repeat the run a few times, adjusting `target_relief` until the **total** relief (channels plus hillslopes) matches the relief we want.

### Edit code block below to change the steady-state landscape!

* `uplift_rate` is the steady-state uplift rate in **m/yr**: 0.0002 m/yr is 0.2 mm/yr. Try values between about 0.00005 (slow) and 0.002 (very fast).
* `total_relief` is the relief of the steady-state landscape (channels plus hillslopes) in metres.
* `hillslope_D` is the hillslope diffusivity in m²/yr. Bigger values give smoother, gentler hillslopes. (With 100 m pixels and channels starting at 20 pixels, hillslopes are only a few pixels long, so the grid shows them only roughly.)

You do not set the erodibility $K$: it is calculated to give your relief, and printed below together with the steepness index $k_{sn}$. Notice that for a fixed relief, a faster uplift rate needs a larger $K$ (more erodible rock), but the channel steepness stays the same: steepness is set by the relief.

In [ ]:
uplift_rate = 0.0002     # m/yr (0.2 mm/yr)
total_relief = 1000      # m, channels plus hillslopes
hillslope_D = 0.1        # m^2/yr

In [ ]:
def write_steady_state_param(target_relief):
    with open(f"{prefix}_ss.param", "w") as f:
        f.write(f"read_fname: {prefix}_initial_filled.tif\n")
        f.write("read_initial_dem: true\n")
        f.write(f"write_fname: {prefix}_ss\n")
        f.write(f"boundary_conditions: {boundary_conditions}\n")
        f.write("run_steady_state: true\n")
        f.write("steady_state_mode: relief\n")
        f.write(f"target_relief: {target_relief}\n")
        f.write("steady_state_n_cycles: 10\n")
        f.write(f"m_exponent: {m_exponent}\n")
        f.write(f"n_exponent: {n_exponent}\n")
        f.write(f"uplift_rate: {uplift_rate}\n")
        # The channel threshold is a specific area (drainage area / pixel width), in metres
        f.write(f"incision_threshold_specific_area: {threshold_pixels * dx}\n")
        f.write("run_hillslope_steady_state: true\n")
        f.write("hillslope_ss_model: linear\n")
        f.write(f"hillslope_diffusivity: {hillslope_D}\n")


# Adjust target_relief (the channel relief) until the total relief is right
channel_relief = total_relief
for attempt in range(1, 9):
    write_steady_state_param(channel_relief)
    !lsdtt-lem ./ {prefix}_ss.param > {prefix}_ss.log
    with rasterio.open(f"{prefix}_ss_hillslope_steady_state_elevation.tif") as src:
        z = src.read(1, masked=True)
    relief_now = float(z.max() - z.min())
    print(f"Attempt {attempt}: channel relief {channel_relief:7.1f} m -> total relief {relief_now:7.1f} m")
    if abs(relief_now - total_relief) < 1:
        break
    channel_relief += total_relief - relief_now

ss_dem = f"{prefix}_ss_hillslope_steady_state_elevation.tif"

stats = pd.read_csv(f"{prefix}_ss_steady_state_stats.csv")
K = float(stats["K_used"].iloc[0])

# k_sn: lsdtt-lem uses specific area A/dx in the stream power law, so for a chi
# plot made with A0 = 1 m^2 the expected gradient is (U/K)^(1/n) * dx^(m/n).
ksn_ss = (uplift_rate / K) ** (1 / n_exponent) * dx ** (m_exponent / n_exponent)

print(f"\nErodibility K       = {K:.3e} m^(1-2m)/yr")
print(f"Steepness index k_sn = {ksn_ss:.1f} m^0.9   (the gradient of the chi plot)")

## Step 3: The steady-state landscape

We analyse the landscape with the same `lsdtt3` tools you would use on real topography:

* `lsdtt-surface-metrics` makes a hillshade and a slope map. (`polyfit_lambda_metres: 250` fits a smooth surface over a window a few pixels across.)
* `lsdtt-channel-extraction` extracts the channel network (here 20 pixels, as in the model) and the drainage area of every pixel.
* `lsdtt-chi-analysis` extracts the channels of the largest basin with their elevation, flow distance and chi.

All three are told about the closed and periodic edges with `boundary_conditions`. We wrap them in a Python function, `analyse`, so that we can reuse it on every snapshot of the transient run in Step 4.

**Choosing the basin.** The basin outlets all sit on the open south edge. `lsdtt-chi-analysis` only analyses *complete* basins, and it counts a basin that touches an open edge as incomplete. So we put the outlet one pixel in from the edge: the pixel in the second-to-last row with the largest drainage area. That is the mouth of the largest basin.

In [ ]:
from pyproj import Transformer


def analyse(dem_fname, tag, outlet=None):
    # Hillshade and slope
    with open(f"{tag}_metrics.param", "w") as f:
        f.write(f"read_fname: {dem_fname}\n")
        f.write(f"write_fname: {tag}\n")
        f.write("write_hillshade_raster: true\n")
        f.write("polyfit_lambda_metres: 250\n")
        f.write("polyfit_metrics: slope\n")
    !lsdtt-surface-metrics ./ {tag}_metrics.param > {tag}_metrics.log

    # Channel network and drainage area
    with open(f"{tag}_channels.param", "w") as f:
        f.write(f"read_fname: {dem_fname}\n")
        f.write(f"write_fname: {tag}\n")
        f.write(f"boundary_conditions: {boundary_conditions}\n")
        f.write("source_extraction_algorithm_choice: threshold\n")
        f.write(f"threshold_contributing_pixels: {threshold_pixels}\n")
        f.write("write_channel_network_lines: true\n")
        f.write("write_drainage_area_raster: true\n")
    !lsdtt-channel-extraction ./ {tag}_channels.param > {tag}_channels.log

    # The outlet of the largest basin: the biggest drainage area one row in from the south edge
    if outlet is None:
        with rasterio.open(f"{tag}_drainage_area.tif") as src:
            area = src.read(1)
            row = area.shape[0] - 2
            col = int(np.argmax(area[row, :]))
            x, y = src.xy(row, col)
            to_ll = Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True)
        lon, lat = to_ll.transform(x, y)
        outlet = (lat, lon)
    pd.DataFrame({"latitude": [outlet[0]], "longitude": [outlet[1]]}).to_csv(
        f"{tag}_outlet.csv", index=False, float_format="%.8f")

    # Chi analysis of the largest basin
    with open(f"{tag}_chi.param", "w") as f:
        f.write(f"read_fname: {dem_fname}\n")
        f.write(f"write_fname: {tag}\n")
        f.write(f"boundary_conditions: {boundary_conditions}\n")
        f.write(f"threshold_contributing_pixels: {threshold_pixels}\n")
        f.write(f"basin_outlet_fname: {tag}_outlet.csv\n")
        f.write("basin_outlet_snapping_method: none\n")
        f.write("n_largest_basins: 0\n")
        f.write(f"m_over_n: {m_exponent / n_exponent}\n")
        f.write("write_chi_data_maps: true\n")
        f.write("chi_data_maps_write_vector_format: csv\n")
    !lsdtt-chi-analysis ./ {tag}_chi.param > {tag}_chi.log

    chi = pd.read_csv(f"{tag}_chi_data_maps.csv")
    return chi, outlet


chi_ss, outlet = analyse(ss_dem, f"{prefix}_ss")
print(f"Largest basin: outlet at latitude {outlet[0]:.5f}, longitude {outlet[1]:.5f}")
print(f"Drainage area {chi_ss.drainage_area.max() / 1e6:.1f} km², {len(chi_ss)} channel pixels")

### Figure 2: Hillshade

The south (open) edge is at the bottom. The map coordinates are those of an arbitrary synthetic grid, in km.

In [ ]:
fig = render_hillshade(f"{prefix}_ss_hillshade.tif", shade=False, style=hs_style)
save_figure(fig, f"{prefix}_ss_hillshade.png", style=style)
display(Image(f"{prefix}_ss_hillshade.png", width=350))

### Figure 3: Channel network

The channels draining at least 20 pixels, thicker and darker for higher Strahler stream orders, over the hillshade. The red circle marks the mouth of the largest basin, the one whose profiles we plot below. Because the east and west edges are periodic, channels that leave one side of the map carry on from the other.

In [ ]:
from lsdviztools3.render.points import render_points

lines = load_vector(f"{prefix}_ss_channel_network_lines.fgb")
outlet_point = gpd.GeoDataFrame(geometry=gpd.points_from_xy([outlet[1]], [outlet[0]]), crs="EPSG:4326")

fig = render_hillshade(f"{prefix}_ss_hillshade.tif", shade=False, style=hs_style)
colours = {1: "lightskyblue", 2: "deepskyblue", 3: "dodgerblue", 4: "blue", 5: "navy", 6: "midnightblue"}
for order in sorted(lines["stream_order"].unique()):
    subset = lines[lines["stream_order"] == order]
    fig = render_channels(subset, fig=fig, style=replace(style, line_width=f"{0.3 * order:.1f}p",
                                                         line_color=colours.get(int(order), "black")))
fig = render_points(outlet_point, symbol="c", pen="1.5p,red", fig=fig, style=replace(style, point_size="0.35c"))
save_figure(fig, f"{prefix}_ss_channels.png", style=style)
display(Image(f"{prefix}_ss_channels.png", width=350))

### Figure 4: Slope

Topographic gradient draped over the hillshade. With linear diffusion the hillslopes are steepest just above the channels and flatten towards the ridges.

In [ ]:
with rasterio.open(f"{prefix}_ss_direct_polyfit_slope.tif") as src:
    slope_max = float(np.percentile(src.read(1, masked=True).compressed(), 99))

fig = render_drape(f"{prefix}_ss_hillshade.tif", f"{prefix}_ss_direct_polyfit_slope.tif",
                   base_is_hillshade=True, transparency=30, cmap="inferno", vmin=0, vmax=slope_max,
                   colorbar=True, colorbar_label="Gradient (m/m)", colorbar_position="right",
                   style=style)
save_figure(fig, f"{prefix}_ss_slope.png", style=style)
display(Image(f"{prefix}_ss_slope.png", width=420))

### Figure 5: Long profile and chi profile of the largest basin

**Chi** ($\chi$) is distance along the channel, measured upstream from the outlet, with each step weighted by drainage area:

$$\chi = \int_{0}^{x} \left(\frac{A_0}{A(x')}\right)^{m/n} dx' .$$

At steady state, elevation is a straight line against chi, and its gradient is $k_{sn}$. The dashed line shows the $k_{sn}$ predicted from the model parameters. The tributaries lie on top of the main stem because we used the model's own $m/n$.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.scatter(chi_ss.flow_distance / 1000, chi_ss.elevation, s=3, c="k")
ax1.set_xlabel("Flow distance from outlet (km)")
ax1.set_ylabel("Elevation (m)")
ax1.set_title("Long profile")

ax2.scatter(chi_ss.chi, chi_ss.elevation, s=3, c="k", label="channel pixels")
chi_line = np.array([0, chi_ss.chi.max()])
ax2.plot(chi_line, chi_ss.elevation.min() + ksn_ss * chi_line, "r--",
         label=f"predicted $k_{{sn}}$ = {ksn_ss:.0f}")
ax2.set_xlabel(r"$\chi$ (m)")
ax2.set_ylabel("Elevation (m)")
ax2.set_title("Chi profile")
ax2.legend()

fig.tight_layout()
fig.savefig(f"{prefix}_ss_profiles.png", dpi=200)
plt.show()

## Step 4: Increase the uplift rate

Now we take the steady-state landscape and run the model **forward in time** with a faster uplift rate, using the same erodibility $K$ and hillslope diffusivity $D$.

The landscape has to steepen until erosion catches up with the new uplift rate. The new steady-state steepness is

$$k_{sn,\text{new}} = \left(\frac{U_\text{new}}{K}\right)^{1/n} ,$$

so with $n = 1$, five times the uplift rate means five times the channel steepness. The extra steepness starts at the outlet and travels up the channel network as a **knickpoint**. Above the knickpoint, the landscape has not yet "heard" about the change.

### Edit code block below to change the uplift increase!

* `uplift_rate_new` is the new uplift rate in m/yr: 0.001 m/yr is 1 mm/yr, five times the steady-state rate. Try a smaller increase (for example 0.0004) or a bigger one (0.002).
* `snapshot_interval_years` and `n_snapshots` set when we look at the landscape: every 200,000 years, five times.
* `timestep` is the model time step in years. Smaller is more accurate but slower.

In [ ]:
uplift_rate_new = 0.001          # m/yr (1 mm/yr)
snapshot_interval_years = 200000
n_snapshots = 5
timestep = 500                   # years

ksn_new = (uplift_rate_new / K) ** (1 / n_exponent) * dx ** (m_exponent / n_exponent)
print(f"Uplift increases by a factor of {uplift_rate_new / uplift_rate:.1f}")
print(f"k_sn: {ksn_ss:.1f} at steady state -> {ksn_new:.1f} at the new steady state")

`lsdtt-lem` reads the steady-state landscape (`read_initial_dem: true`) and runs forward in time:

* `snap_initial_surface_to_steady_state: false`: keep the landscape we give it. (By default the model snaps its starting surface to steady state for the *new* uplift rate, which would skip the transient we want to see.)
* `erodibility` and `hillslope_diffusivity`: the same $K$ and $D$ as the steady state.
* `enable_hillslope_transport: true` with `hillslope_model: adi` and `enable_nonlinear_hillslopes: false`: linear diffusion on the hillslopes, solved implicitly.
* `frequency_of_output_writing`: write the landscape every this many time steps, to `sim_transient_model_iteration_<N>.tif`.
* `steady_state_dz_tolerance: 0`: do not stop early.

This takes a minute or two.

In [ ]:
steps_per_snapshot = int(snapshot_interval_years / timestep)
n_steps = steps_per_snapshot * n_snapshots

with open(f"{prefix}_transient.param", "w") as f:
    f.write(f"read_fname: {ss_dem}\n")
    f.write("read_initial_dem: true\n")
    f.write(f"write_fname: {prefix}_transient\n")
    f.write(f"boundary_conditions: {boundary_conditions}\n")
    f.write("snap_initial_surface_to_steady_state: false\n")
    f.write(f"uplift_rate: {uplift_rate_new}\n")
    f.write(f"erodibility: {K}\n")
    f.write(f"m_exponent: {m_exponent}\n")
    f.write(f"n_exponent: {n_exponent}\n")
    f.write(f"incision_threshold_specific_area: {threshold_pixels * dx}\n")
    f.write("enable_hillslope_transport: true\n")
    f.write("enable_nonlinear_hillslopes: false\n")
    f.write("hillslope_model: adi\n")
    f.write(f"hillslope_diffusivity: {hillslope_D}\n")
    f.write(f"timestep: {timestep}\n")
    f.write(f"number_of_iterations: {n_steps}\n")
    f.write(f"frequency_of_output_writing: {steps_per_snapshot}\n")
    f.write("steady_state_dz_tolerance: 0\n")

!lsdtt-lem ./ {prefix}_transient.param > {prefix}_transient.log

snapshots = [(i * steps_per_snapshot * timestep, f"{prefix}_transient_model_iteration_{i * steps_per_snapshot}.tif")
             for i in range(1, n_snapshots + 1)]
for t, fname in snapshots:
    print(f"{t / 1000:6.0f} kyr: {fname}")

Now we run the same analysis on every snapshot, always using the outlet of the steady-state largest basin, so that we follow the same river through time.

In [ ]:
results = []
for i, (t, fname) in enumerate(snapshots, start=1):
    tag = f"{prefix}_t{i}"
    chi_t, _ = analyse(fname, tag, outlet=outlet)
    results.append((t, tag, fname, chi_t))
    print(f"{t / 1000:6.0f} kyr: analysed, maximum elevation {chi_t.elevation.max():.0f} m in the basin")

### Figure 6: Elevation through time

Elevation draped over the hillshade at each snapshot. All the maps share one colour scale, so you can see the landscape gaining relief as the uplift outpaces erosion. Watch the valleys deepen and the steepening work its way north from the open edge.

In [ ]:
def show_series(layer, cmap, label, vmin, vmax, fname, drape=False):
    # One map per snapshot, all on the same colour scale. layer is "elevation" or the
    # suffix of an lsdtt-surface-metrics output. With drape=True the layer is drawn
    # semi-transparently over the hillshade.
    fig, axes = plt.subplots(1, n_snapshots, figsize=(3.5 * n_snapshots, 6), sharey=True)
    for ax, (t, tag, dem, _) in zip(np.atleast_1d(axes), results):
        path = dem if layer == "elevation" else f"{tag}_{layer}.tif"
        with rasterio.open(path) as src:
            data = src.read(1, masked=True)
            b = src.bounds
        extent = [0, (b.right - b.left) / 1000, 0, (b.top - b.bottom) / 1000]
        if drape:
            with rasterio.open(f"{tag}_hillshade.tif") as src:
                ax.imshow(src.read(1, masked=True), cmap="gray", extent=extent)
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, extent=extent,
                       alpha=0.6 if drape else 1.0)
        ax.set_title(f"{t / 1000:.0f} kyr")
        ax.set_xlabel("km")
    np.atleast_1d(axes)[0].set_ylabel("km")
    cbar = fig.colorbar(im, ax=axes, shrink=0.7, label=label)
    cbar.solids.set_alpha(1)
    fig.savefig(fname, dpi=200, bbox_inches="tight")
    plt.show()


# One colour scale for every snapshot: from 0 m to the highest point in any of them
elev_max = 0
for _, _, dem, _ in results:
    with rasterio.open(dem) as src:
        elev_max = max(elev_max, float(src.read(1, masked=True).max()))

show_series("elevation", "gist_earth", "Elevation (m)", 0, elev_max,
            f"{prefix}_transient_elevation.png", drape=True)

### Figure 7: Slope maps through time

All the maps share one colour scale. The steepened, rapidly eroding part of the landscape spreads from the south edge.

In [ ]:
with rasterio.open(f"{results[-1][1]}_direct_polyfit_slope.tif") as src:
    slope_max_t = float(np.percentile(src.read(1, masked=True).compressed(), 99))

show_series("direct_polyfit_slope", "inferno", label="Gradient (m/m)", vmin=0, vmax=slope_max_t,
            fname=f"{prefix}_transient_slopes.png")

### Figure 8: Long profiles and chi profiles through time

The profiles of the largest basin at steady state (black) and at each snapshot (coloured by time). The dashed lines show the old and the new steady-state $k_{sn}$.

In the chi plot, look for the **knickpoint**: the kink between the steep lower segment, which has adjusted to the new uplift rate, and the gentler upper segment, which still has the old steepness. Because the knickpoint moves upstream at a speed that depends on drainage area, it reaches the same chi value in every tributary at the same time.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
colours_t = plt.cm.viridis(np.linspace(0, 0.9, n_snapshots))

for ax, xcol, xscale in [(ax1, "flow_distance", 1000), (ax2, "chi", 1)]:
    ax.scatter(chi_ss[xcol] / xscale, chi_ss.elevation, s=2, c="k", label="steady state")
    for (t, tag, _, chi_t), colour in zip(results, colours_t):
        ax.scatter(chi_t[xcol] / xscale, chi_t.elevation, s=2, color=colour, label=f"{t / 1000:.0f} kyr")
    ax.set_ylabel("Elevation (m)")

chi_line = np.array([0, chi_ss.chi.max()])
ax2.plot(chi_line, ksn_ss * chi_line, "k--", lw=1, label=f"old $k_{{sn}}$ = {ksn_ss:.0f}")
ax2.plot(chi_line, ksn_new * chi_line, "r--", lw=1, label=f"new $k_{{sn}}$ = {ksn_new:.0f}")
ax2.set_ylim(0, 1.05 * max(chi_t.elevation.max() for _, _, _, chi_t in results))

ax1.set_xlabel("Flow distance from outlet (km)")
ax1.set_title("Long profiles")
ax2.set_xlabel(r"$\chi$ (m)")
ax2.set_title("Chi profiles")
ax2.legend(markerscale=4, fontsize=8)

fig.tight_layout()
fig.savefig(f"{prefix}_transient_profiles.png", dpi=200)
plt.show()

## Step 5: Swath profiles of the final landscape

A **swath profile** summarises all the elevations in a strip either side of a line (the *baseline*): at each distance along the line it shows the range of elevations (minimum to maximum), the middle half (25th to 75th percentile), the median and the mean. We used one in the transience notebook. Here we draw four swaths across the final landscape (1000 kyr):

1. **North–south, whole model.** The baseline runs up the middle of the model from the open south edge to the closed north edge, and the swath is as wide as the model, so it includes every pixel.
2. **Along the main channel.** The baseline follows the longest channel in the largest basin, from its mouth to its source. The channel is a staircase of D8 pixel steps, and every kink in a baseline muddles which pixels belong to which bin, so we smooth the channel with a moving average before using it.
3. **East–west, near the north edge.** Across the whole model, close to the closed north edge, where the landscape has not yet felt the uplift increase.
4. **East–west, near the south edge.** Across the whole model, close to the open south edge, where the steepening started.

The last three are 30 pixels wide (15 pixels either side of the baseline).

### Edit code block below to change the swaths!

* `swath_width_pixels` is the full width of the main-channel and east–west swaths.
* `ew_distance_from_edge_pixels` is how far in from the north and south edges the east–west baselines are.
* `swath_bin_width_metres` sets how finely each swath is binned along its baseline.
* `channel_smoothing_pixels` is the length of the moving average used to smooth the main channel, in channel pixels. Bigger is smoother.

In [ ]:
swath_width_pixels = 30
ew_distance_from_edge_pixels = 20
swath_bin_width_metres = 200
channel_smoothing_pixels = 21

In [ ]:
from shapely.geometry import LineString
from lsdviztools3.io.swath import load_swath
from lsdviztools3.charts import plot_swath

final_time, final_tag, final_dem, final_chi = results[-1]
with rasterio.open(final_dem) as src:
    b = src.bounds
    dem_crs = src.crs

# The main channel: the pixels of the longest channel in the basin, from the mouth upstream
longest = final_chi.loc[final_chi.flow_distance.idxmax(), "source_key"]
main_stem = final_chi[final_chi.source_key == longest].sort_values("flow_distance")
main_xy = gpd.GeoSeries(gpd.points_from_xy(main_stem.longitude, main_stem.latitude),
                        crs="EPSG:4326").to_crs(dem_crs)
# Smooth with a moving average, keeping the two ends where they are
smooth = lambda v: pd.Series(np.asarray(v)).rolling(channel_smoothing_pixels, center=True,
                                                    min_periods=1).mean().to_numpy().copy()
xs, ys = smooth(main_xy.x), smooth(main_xy.y)
xs[[0, -1]] = np.asarray(main_xy.x)[[0, -1]]
ys[[0, -1]] = np.asarray(main_xy.y)[[0, -1]]
main_line = LineString(zip(xs, ys))

x_mid = (b.left + b.right) / 2
y_south = b.bottom + ew_distance_from_edge_pixels * dx
y_north = b.top - ew_distance_from_edge_pixels * dx
half_width = swath_width_pixels * dx / 2

# name: (title, baseline, half width in metres)
swaths = {
    "ns_full": ("North–south, whole model",
                LineString([(x_mid, b.bottom + dx / 2), (x_mid, b.top - dx / 2)]), (b.right - b.left) / 2),
    "main_channel": ("Along the main channel", main_line, half_width),
    "ew_north": ("East–west, near the north edge",
                 LineString([(b.left + dx / 2, y_north), (b.right - dx / 2, y_north)]), half_width),
    "ew_south": ("East–west, near the south edge",
                 LineString([(b.left + dx / 2, y_south), (b.right - dx / 2, y_south)]), half_width),
}

for name, (title, line, hw) in swaths.items():
    gpd.GeoDataFrame({"name": [name]}, geometry=[line], crs=dem_crs).to_file(f"{prefix}_swath_{name}_baseline.fgb")
    with open(f"{prefix}_swath_{name}.param", "w") as f:
        f.write(f"read_fname: {final_dem}\n")
        f.write(f"write_fname: {prefix}_swath_{name}\n")
        f.write(f"baseline_vector_fname: {prefix}_swath_{name}_baseline.fgb\n")
        f.write(f"half_width_metres: {hw}\n")
        f.write(f"bin_width_metres: {swath_bin_width_metres}\n")
    !lsdtt-swath-profile ./ {prefix}_swath_{name}.param > {prefix}_swath_{name}.log
    print(f"{title}: baseline {line.length / 1000:.1f} km long, {2 * hw / 1000:.1f} km wide")

### Figure 9: Where the swaths are

The final landscape, with each baseline (solid line) and the edge of its swath (dashed). The north–south swath covers the whole model, so its edges are the edges of the map.

In [ ]:
with rasterio.open(f"{final_tag}_hillshade.tif") as src:
    hs = src.read(1, masked=True)
with rasterio.open(final_dem) as src:
    elev = src.read(1, masked=True)

km = lambda x, y: ((np.asarray(x) - b.left) / 1000, (np.asarray(y) - b.bottom) / 1000)
extent = [0, (b.right - b.left) / 1000, 0, (b.top - b.bottom) / 1000]

fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(hs, cmap="gray", extent=extent)
ax.imshow(elev, cmap="gist_earth", alpha=0.5, extent=extent)
swath_colours = {"ns_full": "cyan", "main_channel": "red", "ew_north": "orange", "ew_south": "magenta"}
for name, (title, line, hw) in swaths.items():
    colour = swath_colours[name]
    ax.plot(*km(*line.xy), color=colour, lw=2, label=title)
    if name != "ns_full":
        outline = line.buffer(hw, cap_style="flat").exterior
        ax.plot(*km(*outline.xy), color=colour, lw=1, ls="--")
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])
ax.set_xlabel("km")
ax.set_ylabel("km")
ax.set_title(f"Swaths on the landscape at {final_time / 1000:.0f} kyr")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, fontsize=8)
fig.savefig(f"{prefix}_swath_map.png", dpi=200, bbox_inches="tight")
plt.show()

### Figure 10: The swath profiles

Distance is measured along each baseline from its start: from the open south edge for the north–south and main-channel swaths, and from the west edge for the east–west swaths.

**Questions:**

* In the north–south swath, where does the steep zone end? Compare it with the knickpoint in Figure 8.
* Compare the two east–west swaths. Which has the larger relief between valley floors and ridges, and why?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (name, (title, line, hw)) in zip(axes.ravel(), swaths.items()):
    swath = load_swath(f"{prefix}_swath_{name}_swath_profile.csv")
    plot_swath(swath, bands=((0, 100), (25, 75)), distance_units="km", ax=ax)
    ax.set_title(title)
fig.tight_layout()
fig.savefig(f"{prefix}_swath_profiles.png", dpi=200)
plt.show()

## Things to try

* **Uplift rate at steady state.** Change `uplift_rate` in Step 2 but keep `total_relief`. How does $K$ change? Does the landscape look different?
* **Relief.** Change `total_relief`. How do $K$ and $k_{sn}$ change?
* **Size of the uplift increase.** Change `uplift_rate_new` in Step 4. How fast does the knickpoint move, and does it depend on the size of the increase? (With $n = 1$ it should not: the speed is set by $K$ and drainage area alone.)
* **Hillslopes.** Change `hillslope_D`. How do the hillslopes respond to the incision, compared with the channels?
* **A different landscape.** Change `random_seed` in Step 1, or set `valley_slope = 0` to let the noise alone decide where the big rivers go.

## Citing lsdtt3

You can find the citation for `lsdtt3` by running any of its programs with `-c`, for example `!lsdtt-lem -c`.